# Face Recognition Open-Set dengan FaceNet dan ChromaDB

Notebook ini melakukan enrollment embedding wajah, pencarian nearest neighbor di ChromaDB persistent, dan evaluasi threshold. Tidak ada training atau fine-tuning backbone.

## 1. Setup environment

Jalankan cell ini sekali setelah runtime Colab baru dimulai.

In [ ]:
# Jalankan cell ini sekali di runtime Colab baru.
# facenet-pytorch dipasang tanpa dependency agar Torch/Numpy bawaan Colab tidak diganti.
%pip install -q --no-cache-dir "chromadb==0.5.23"
%pip install -q --no-cache-dir --no-deps "facenet-pytorch==2.6.0"

print('Instalasi selesai. Jika Colab meminta restart, pilih Runtime > Restart session, lalu lanjutkan ke cell berikutnya.')

In [ ]:
from pathlib import Path
import hashlib

import chromadb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from facenet_pytorch import InceptionResnetV1, MTCNN
from PIL import Image
from tqdm.auto import tqdm

# Dataset harus sudah di-upload ke runtime dan berada di /content/data.
ROOT_DIR = Path('/content/data')
DATASET_DIR = ROOT_DIR
TEST_DIR = ROOT_DIR / 'test_photos'
CHROMA_DIR = Path('/content/face_chroma_db')
OUTPUT_DIR = Path('/content/face_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Dataset: {DATASET_DIR}')
if not DATASET_DIR.exists():
    raise FileNotFoundError(f'Dataset tidak ditemukan: {DATASET_DIR}')
person_dirs = [
    path for path in DATASET_DIR.iterdir()
    if path.is_dir() and path.name not in {'test_photos', 'chroma_db', 'outputs'}
]
if not person_dirs:
    raise FileNotFoundError(f'Tidak ada folder orang di {DATASET_DIR}')
print(f'Jumlah orang: {len(person_dirs)}')
print('Nama folder:', ', '.join(sorted(path.name for path in person_dirs)))

In [ ]:
# MTCNN melakukan deteksi/crop, ResNet menghasilkan embedding 512 dimensi.
mtcnn = MTCNN(
    image_size=160,
    margin=0,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=True,
    keep_all=False,
    device=DEVICE,
)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
print('Model siap.')

## 2. Persistent ChromaDB

Folder `face_chroma_db` dibuat di runtime Colab. Database ini tersedia selama runtime aktif; download sebagai ZIP setelah enrollment jika ingin menyimpannya. Jalankan cell ini sebelum enrollment atau inference.

In [ ]:
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(
    name='faces',
    metadata={'hnsw:space': 'cosine'},
)
print(f'ChromaDB: {CHROMA_DIR}')
print(f'Jumlah embedding tersimpan: {collection.count()}')

## 3. Helper embedding dan enrollment

Setiap subfolder langsung di bawah `/content/data/` dianggap sebagai label. Folder `chroma_db`, `outputs`, dan `test_photos` otomatis dikecualikan.

In [ ]:
EXCLUDED_DIRS = {'chroma_db', 'outputs', 'test_photos'}

def image_embedding(image_path):
    """Detect one face and return its L2-normalized embedding, or None."""
    try:
        with Image.open(image_path) as image:
            face = mtcnn(image.convert('RGB'))
        if face is None:
            return None
        with torch.no_grad():
            embedding = resnet(face.unsqueeze(0).to(DEVICE))
            embedding = F.normalize(embedding, p=2, dim=1)
        return embedding[0].cpu().numpy().astype(np.float32).tolist()
    except (OSError, ValueError, RuntimeError) as error:
        print(f'Gagal memproses {image_path}: {error}')
        return None

def embedding_id(person, image_path):
    """Create a stable Chroma ID from the image path."""
    relative_path = image_path.relative_to(DATASET_DIR).as_posix()
    return hashlib.sha1(relative_path.encode('utf-8')).hexdigest()

def dataset_images():
    """Yield (person, image path) pairs from enrollment folders."""
    for person_dir in sorted(DATASET_DIR.iterdir()):
        if not person_dir.is_dir() or person_dir.name in EXCLUDED_DIRS:
            continue
        for image_path in sorted(person_dir.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                yield person_dir.name, image_path

def enroll_dataset():
    """Embed all enrollment images and upsert them into persistent ChromaDB."""
    records = []
    skipped = []
    people = set()
    image_pairs = list(dataset_images())

    for person, image_path in tqdm(image_pairs, desc='Enrollment'):
        people.add(person)
        vector = image_embedding(image_path)
        if vector is None:
            skipped.append({'person': person, 'source_file': str(image_path), 'reason': 'no_face_or_invalid_image'})
            continue
        records.append({
            'id': embedding_id(person, image_path),
            'embedding': vector,
            'metadata': {'person': person, 'source_file': str(image_path)},
        })

    if records:
        collection.upsert(
            ids=[record['id'] for record in records],
            embeddings=[record['embedding'] for record in records],
            metadatas=[record['metadata'] for record in records],
        )

    skipped_path = OUTPUT_DIR / 'skipped_files.csv'
    pd.DataFrame(skipped, columns=['person', 'source_file', 'reason']).to_csv(skipped_path, index=False)
    summary = pd.DataFrame([{
        'total_persons': len(people),
        'total_images_found': len(image_pairs),
        'total_images_embedded': len(records),
        'total_images_skipped': len(skipped),
        'chroma_collection_count': collection.count(),
    }])
    summary.to_csv(OUTPUT_DIR / 'enrollment_summary.csv', index=False)
    print(summary.to_string(index=False))
    if skipped:
        print('File yang di-skip:')
        display(pd.DataFrame(skipped))
    return summary, skipped

summary, skipped = enroll_dataset()

## 4. Inference / verification

`confidence` di sini adalah cosine similarity, bukan probabilitas. Nilai tetap dikembalikan untuk kasus `unknown`.

In [ ]:
def verify(image_path, threshold=0.6, top_k=3):
    """Compare an image with enrolled faces and return an open-set result."""
    image_path = Path(image_path)
    if not image_path.is_file():
        return {'status': 'error', 'message': f'File tidak ditemukan: {image_path}'}
    vector = image_embedding(image_path)
    if vector is None:
        return {'status': 'no_face_detected'}
    if collection.count() == 0:
        return {'status': 'error', 'message': 'ChromaDB masih kosong. Jalankan enrollment terlebih dahulu.'}

    result = collection.query(
        query_embeddings=[vector],
        n_results=min(top_k, collection.count()),
        include=['distances', 'metadatas'],
    )
    distances = result['distances'][0]
    metadatas = result['metadatas'][0]
    similarities = [1.0 - float(distance) for distance in distances]
    best_similarity = similarities[0]
    best_person = metadatas[0]['person']
    response = {
        'status': 'recognized' if best_similarity >= threshold else 'unknown',
        'confidence': round(best_similarity, 6),
        'threshold': threshold,
        'closest_match': best_person,
        'source_file': metadatas[0].get('source_file'),
        'top_matches': [
            {'person': metadata['person'], 'similarity': round(similarity, 6), 'source_file': metadata.get('source_file')}
            for metadata, similarity in zip(metadatas, similarities)
        ],
    }
    if response['status'] == 'recognized':
        response['person'] = best_person
    return response

# Contoh penggunaan:
# verify('/content/data/nama_orang/foto.jpg')

## 5. Evaluasi threshold

Buat folder opsional berikut di runtime. Nama subfolder menjadi ground truth; folder `unknown` dipakai untuk wajah yang tidak terdaftar:

```text
/content/data/test_photos/budi/foto_uji.jpg
/content/data/test_photos/sari/foto_uji.jpg
/content/data/test_photos/unknown/orang_lain.jpg
```

In [ ]:
def evaluation_images():
    """Yield test images and their optional folder-based ground truth."""
    if not TEST_DIR.exists():
        return
    for path in sorted(TEST_DIR.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        relative = path.relative_to(TEST_DIR)
        ground_truth = relative.parts[0] if len(relative.parts) > 1 else None
        if ground_truth == 'unknown':
            ground_truth = None
        yield path, ground_truth

def evaluate(threshold=0.6):
    """Run verification for test images and save tabular evaluation results."""
    rows = []
    for image_path, ground_truth in tqdm(list(evaluation_images()), desc='Evaluation'):
        result = verify(image_path, threshold=threshold)
        predicted = result.get('person') or result.get('closest_match')
        confidence = result.get('confidence')
        status = result.get('status')
        is_correct = None
        if ground_truth is not None:
            is_correct = status == 'recognized' and predicted == ground_truth
        elif status == 'unknown':
            is_correct = True
        rows.append({
            'file': str(image_path),
            'predicted_person': predicted,
            'status': status,
            'confidence': confidence,
            'ground_truth': ground_truth or 'unknown',
            'is_correct': is_correct,
        })
    results = pd.DataFrame(rows)
    results.to_csv(OUTPUT_DIR / 'evaluation_results.csv', index=False)
    if results.empty:
        print(f'Tidak ada foto uji di {TEST_DIR}')
        return results
    display(results)
    return results

evaluation_results = evaluate(threshold=0.6)

In [ ]:
def plot_confidence_distribution(results):
    """Plot similarity distributions for correct and incorrect known-person predictions."""
    if results.empty or 'confidence' not in results:
        print('Tidak ada hasil untuk diplot.')
        return
    known = results[results['ground_truth'] != 'unknown'].copy()
    correct = known[known['is_correct'] == True]['confidence'].dropna()
    incorrect = known[known['is_correct'] == False]['confidence'].dropna()
    unknown = results[results['ground_truth'] == 'unknown']['confidence'].dropna()
    plt.figure(figsize=(9, 5))
    if not correct.empty:
        plt.hist(correct, bins=15, alpha=0.65, label='Match benar')
    if not incorrect.empty:
        plt.hist(incorrect, bins=15, alpha=0.65, label='Match salah')
    if not unknown.empty:
        plt.hist(unknown, bins=15, alpha=0.65, label='Ground truth unknown')
    plt.axvline(0.6, color='black', linestyle='--', label='Threshold 0.6')
    plt.xlabel('Cosine similarity')
    plt.ylabel('Jumlah foto')
    plt.title('Distribusi confidence')
    plt.legend()
    plt.tight_layout()
    plot_path = OUTPUT_DIR / 'confidence_distribution.png'
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f'Plot disimpan: {plot_path}')

plot_confidence_distribution(evaluation_results)

## Output dan integrasi serving

Output utama berada di:

```text
/content/face_chroma_db/                  # database persistent selama runtime
/content/face_outputs/enrollment_summary.csv
/content/face_outputs/skipped_files.csv
/content/face_outputs/evaluation_results.csv
/content/face_outputs/confidence_distribution.png
```

Pada FastAPI nanti, load `MTCNN`, `InceptionResnetV1`, `PersistentClient(path=CHROMA_DIR)`, dan fungsi `verify()` sekali saat startup. Endpoint upload hanya memanggil `verify()`; enrollment tidak dijalankan pada setiap request. Download database dengan ZIP sebelum runtime dihapus.

## 6. Download backup ChromaDB

Jalankan setelah enrollment selesai. Seluruh folder database dikemas sebagai ZIP agar bisa dipakai kembali di FastAPI atau runtime lain.

In [ ]:
import shutil

zip_path = shutil.make_archive(
    base_name='/content/face_chroma_db_backup',
    format='zip',
    root_dir=CHROMA_DIR.parent,
    base_dir=CHROMA_DIR.name,
)
print(f'Backup dibuat: {zip_path}')

from google.colab import files
files.download(zip_path)